In [ ]:
df = None

In [ ]:
#               ┌────────────────────── Потребность в масштабировании ──────────────────────┐
#               │                                                                           │
#  Требуют масштабирования (Чувствительны)                                    НЕ требуют масштабирования
#   ┌───────────────────────┼───────────────────────┐                                    ┌────────────┴────────────┐
#   ▼                       ▼                       ▼                                    ▼                         ▼
# Метрические             Линейные              Нейросети                            Деревья решений           Ансамбли
# (KNN, KMeans, SVM)    (Linear/Logistic)      (Deep Learning)                      (Decision Tree)         (Random Forest, XGBoost)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# нет аномалий
scaler = MinMaxScaler()
df['Age_minmax'] = scaler.fit_transform(df[['Age']])

In [ ]:
from sklearn.preprocessing import StandardScaler
# почти нет аномалий
scaler = StandardScaler()
df['Age_std'] = scaler.fit_transform(df[['Age']])

In [ ]:
from sklearn.preprocessing import RobustScaler
# доху..(много) аномалий
scaler = RobustScaler()
df['Income_robust'] = scaler.fit_transform(df[['Income']])

In [ ]:
#                    ┌────────────────────── Правильный workflow ──────────────────────┐
#                    │                             │                                   │
#  1. Train / Test Split                    2. Fit на Train                    3. Transform на обоих
#   Сначала делим данные на               scaler.fit(X_train)                X_train_scaled = scaler.transform(X_train)
#   обучающую и тестовую часть            Считаем min/max/среднее            X_test_scaled = scaler.transform(X_test)
#                                         ИСКЛЮЧИТЕЛЬНО по Train!            Используем статистики от TRAIN!

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
#           ┌───────────────────── Типы категориальных признаков ─────────────────────┐
#           │                                                                         │
#       ▼ Порядковые (Ordinal)                                                    ▼ Номинальные (Nominal)
#   Есть естественный порядок или иерархия.                                   НЕТ никакого порядка или иерархии.
#   (e.g., "Низкий" < "Средний" < "Высокий")                                  (e.g., "Город", "Цвет", "Марка машины")

In [ ]:
# 1. Label / Ordinal Encoding (Порядковое кодирование)
from sklearn.preprocessing import OrdinalEncoder

# явно задаём порядок категорий
encoder = OrdinalEncoder(categories=[['Junior', 'Middle', 'Senior']])
df['Level_encoded'] = encoder.fit_transform(df[['Level']])

In [ ]:
# 2. One-Hot Encoding / OHE (Прямое кодирование / Дамми-кодирование)
# если уникальных значений у категории меньше 10-15
# каждому значению по столбцу с 1 или 0

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Способ 1: Через Scikit-Learn (рекомендуется в ML-пайплайнах)
ohe = OneHotEncoder(sparse_output=False, drop='first')
city_encoded = ohe.fit_transform(df[['City']])

# Способ 2: Быстро в Pandas (для EDA)
df_ohe = pd.get_dummies(df, columns=['City'], drop_first=True)

In [ ]:
# 3. Две классические ловушки OHE

# Ловушка 1: Мультиколлинеарность (Dummy Variable Trap)
# Обрати внимание на таблицу OHE выше: 
# если мы знаем, что City_MSK = 0 и City_SPB = 0, то мы на 100% уверены, 
# что City_KZN = 1! Последний столбец линейно выражается через предыдущие.
# В чём проблема: Для линейных моделей линейная зависимость между признаками 
# (мультиколлинеарность) критична — матрица X.T@X становится необратимой 
# или нестабильной, а веса улетают в бесконечность.
# Решение: Всегда удаляем один (первый) базовый столбец! 
# В Scikit-Learn за это отвечает параметр drop='first' 
# (или drop_first=True в Pandas). Для N категорий мы создаем N-1 столбцов.K

# Ловушка 2: если уникальных значений очень много 500+
# нужно использовать другие алгоритмы
# пройдём их позже

In [ ]:
# Target Encoding
# заменяет категорию на среднее в таргет категории (район - цена квартиры)

# Если делать Target Encoding влоб 
# (просто через groupby('category')['target'].mean()), 
# модель мгновенно переобучится.

# решение сглаживание, чтобы редкие категории не получали 
# экстремальные значения, подмешиваем к ним среднее во всем данным
# ФОРМУЛА В ТЕТРАДИ

